# Notebook 05: Complete GPT Model

## Overview

- **Duration**: ~2.5 hours
- **Prerequisites**: Notebooks 01-04
- **Learning Objectives**:
  1. Understand GPT architecture end-to-end
  2. Stack transformer blocks
  3. Implement the language model head
  4. Initialize weights properly
  5. Understand weight tying

## Introduction

GPT (Generative Pre-trained Transformer) is a **decoder-only** transformer that predicts the next token given a sequence.

### Architecture

```
Input IDs -> Token Embed + Pos Embed -> [Transformer Block] x N -> LayerNorm -> LM Head -> Logits
```

In [1]:
import sys
sys.path.insert(0, '../..')

import math
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

print(f"Using device: {device}")

Using device: cpu


## Import Components from Previous Notebooks

In [2]:
# Import from shared module (or define inline), you can save all models in a folder "shared", then call them.
try:
    from shared.model import LayerNorm, CausalSelfAttention, FeedForward, TransformerBlock
    print("Imported components from shared module")
except ImportError:
    print("Defining components inline...")
    
    class LayerNorm(nn.Module):
        def __init__(self, ndim, bias=True, eps=1e-5):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(ndim))
            self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
            self.eps = eps
        def forward(self, x):
            mean = x.mean(-1, keepdim=True)
            var = x.var(-1, keepdim=True, unbiased=False)
            out = self.weight * (x - mean) / (var + self.eps).sqrt()
            return out + self.bias if self.bias is not None else out
    
    class CausalSelfAttention(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.n_head = config.n_head
            self.n_embd = config.n_embd
            self.head_dim = config.n_embd // config.n_head
            self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
            self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
            self.attn_dropout = nn.Dropout(config.dropout)
            self.resid_dropout = nn.Dropout(config.dropout)
            mask = torch.tril(torch.ones(config.block_size, config.block_size))
            self.register_buffer('mask', mask.view(1, 1, config.block_size, config.block_size))
        def forward(self, x):
            B, T, C = x.size()
            qkv = self.c_attn(x)
            q, k, v = qkv.split(self.n_embd, dim=2)
            q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
            att = att.masked_fill(self.mask[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.resid_dropout(self.c_proj(y))
    
    class FeedForward(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
            self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
            self.dropout = nn.Dropout(config.dropout)
        def forward(self, x):
            return self.dropout(self.c_proj(F.gelu(self.c_fc(x))))
    
    class TransformerBlock(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
            self.attn = CausalSelfAttention(config)
            self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
            self.mlp = FeedForward(config)
        def forward(self, x):
            x = x + self.attn(self.ln_1(x))
            x = x + self.mlp(self.ln_2(x))
            return x

Defining components inline...


## Exercise 5.1: GPT Configuration

Create a configuration dataclass to hold model hyperparameters.

In [3]:
####define configuration
@dataclass
class GPTConfig:
    """Configuration for GPT model."""
    
    # TODO: Define the configuration parameters
    # vocab_size: int = 50257  # GPT-2 vocabulary size
    # block_size: int = 1024   # Maximum sequence length
    # n_layer: int = 12        # Number of transformer blocks
    # n_head: int = 12         # Number of attention heads
    # n_embd: int = 768        # Embedding dimension
    # dropout: float = 0.1     # Dropout probability
    # bias: bool = False       # Use bias in linear layers?
#for example that we copy the default parameters    
    vocab_size: int = 50257
    block_size: int = 1024
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.1
    bias: bool = False

In [4]:
# Test configuration
config = GPTConfig()
print(f"Vocabulary size: {config.vocab_size}")
print(f"Block size: {config.block_size}")
print(f"Number of layers: {config.n_layer}")
print(f"Number of heads: {config.n_head}")
print(f"Embedding dimension: {config.n_embd}")

Vocabulary size: 50257
Block size: 1024
Number of layers: 12
Number of heads: 12
Embedding dimension: 768


## Exercise 5.2: Complete GPT Model (60 min)

Implement the full GPT model.

In [ ]:
####solution 5.2
class GPT(nn.Module):
    """
    GPT Language Model.
    
    Architecture:
        Input IDs -> Token Embed + Pos Embed -> Dropout
        -> [TransformerBlock] x n_layer
        -> LayerNorm -> LM Head -> Logits
    """
    
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        
        # TODO: Create model components
        # 
        # self.transformer = nn.ModuleDict(dict(
        #     wte = ???,  # Token embeddings: nn.Embedding(vocab_size, n_embd)
        #     wpe = ???,  # Position embeddings: nn.Embedding(block_size, n_embd)
        #     drop = ???, # Dropout
        #     h = ???,    # Transformer blocks: nn.ModuleList
        #     ln_f = ???, # Final LayerNorm
        # ))
        # 
        # self.lm_head = ???  # Linear(n_embd, vocab_size, bias=False)
        #
        # Weight tying: share weights between token embeddings and lm_head
        # self.transformer.wte.weight = self.lm_head.weight
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))

        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def _init_weights(self, module):
    # TODO: Implement weight initialization
    # Hint: Use isinstance() to check module type
    # For residual projections (c_proj), use smaller std
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        # For c_proj layers in attention and feedforward, use smaller std
        if isinstance(module, CausalSelfAttention):
            nn.init.normal_(module.c_proj.weight, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))
            if module.c_proj.bias is not None:
                nn.init.zeros_(module.c_proj.bias)
        if isinstance(module, FeedForward):
            nn.init.normal_(module.c_proj.weight, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))
            if module.c_proj.bias is not None:
                nn.init.zeros_(module.c_proj.bias)
    
    def forward(self, idx, targets=None):
        
        # TODO: Implement forward pass
        # 1. Get batch size and sequence length
        """PyTorch handles batching automatically through broadcasting. The model doesn't need to explicitly reference batch_size because:

            Token and position embeddings work across batches automatically
            Transformer blocks operate element-wise and preserve batch dimensions
            The loss computation uses .view(-1, ...) which flattens the batch dimension automatically
            
            can be simplified to:
            seq_len = idx.size(1)"""
        batch_size, seq_len = idx.size()
        # 2. Create position indices [0, 1, 2, ..., seq_len-1]
        positions = torch.arange(seq_len, dtype=torch.long, device=idx.device)
        # 3. Get token embeddings: wte(idx)
        tok_emb = self.transformer.wte(idx)
        # 4. Get position embeddings: wpe(positions)
        pos_emb = self.transformer.wpe(positions)
        # 5. Combine and apply dropout: drop(tok_emb + pos_emb)
        x = self.transformer.drop(tok_emb + pos_emb)
        # 6. Pass through all transformer blocks
        for block in self.transformer.h:
            x = block(x)
        # 7. Apply final layer norm
        x = self.transformer.ln_f(x)
        # 8. Get logits: lm_head(x)
        logits = self.lm_head(x)
        # 9. If targets provided, compute cross-entropy loss
        loss = None
        if targets is not None:
            # Shift logits and targets for next-token prediction
            logits = logits[..., :-1, :].contiguous()
            targets = targets[..., 1:].contiguous()
            # Compute cross-entropy loss
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        
        return logits, loss

In [10]:
####test cell
# Test with a small configuration
small_config = GPTConfig(
    vocab_size=1000,
    block_size=256,
    n_layer=4,
    n_head=4,
    n_embd=128,
    dropout=0.1,
    bias=False,
)

model = GPT(small_config)
print(f"Model created with {sum(p.numel() for p in model.parameters()):,} parameters")

Model created with 948,352 parameters


In [11]:
####test cell
# Test forward pass
batch_size = 2
seq_len = 32

idx = torch.randint(0, small_config.vocab_size, (batch_size, seq_len))
logits, loss = model(idx)

print(f"Input shape: {idx.shape}")
print(f"Logits shape: {logits.shape}")
print(f"Loss: {loss}")

assert logits.shape == (batch_size, seq_len, small_config.vocab_size)
print("\n✓ Forward pass works!")

Input shape: torch.Size([2, 32])
Logits shape: torch.Size([2, 32, 1000])
Loss: None

✓ Forward pass works!


In [12]:
####test cell
# Test with targets (for training)
targets = torch.randint(0, small_config.vocab_size, (batch_size, seq_len))
logits, loss = model(idx, targets)

print(f"Loss with targets: {loss.item():.4f}")
print(f"Expected initial loss (random): ~{math.log(small_config.vocab_size):.4f}")

# Loss should be close to -log(1/vocab_size) for random initialization
expected_loss = math.log(small_config.vocab_size)
assert abs(loss.item() - expected_loss) < 1.0, "Initial loss seems off"
print("\n✓ Loss computation works!")

Loss with targets: 7.0191
Expected initial loss (random): ~6.9078

✓ Loss computation works!


## Exercise 5.3: Parameter Counting (15 min)

Analyze where the parameters are in the model.

In [13]:
def count_parameters(model: nn.Module) -> dict:
    """Count parameters by component."""
    counts = {}
    
    for name, param in model.named_parameters():
        # Get top-level component name
        component = name.split('.')[1] if '.' in name else name
        counts[component] = counts.get(component, 0) + param.numel()
    
    return counts

param_counts = count_parameters(model)
total = sum(param_counts.values())

print("Parameter distribution:")
for component, count in sorted(param_counts.items(), key=lambda x: -x[1]):
    print(f"  {component}: {count:,} ({count/total*100:.1f}%)")
print(f"\nTotal: {total:,}")

Parameter distribution:
  h: 787,456 (83.0%)
  wte: 128,000 (13.5%)
  wpe: 32,768 (3.5%)
  ln_f: 128 (0.0%)

Total: 948,352


## Model Size Configurations

In [14]:
# Different model sizes for experimentation
CONFIGS = {
    'tiny': GPTConfig(vocab_size=10000, block_size=256, n_layer=4, n_head=4, n_embd=128),
    'small': GPTConfig(vocab_size=10000, block_size=512, n_layer=6, n_head=6, n_embd=384),
    'medium': GPTConfig(vocab_size=10000, block_size=512, n_layer=8, n_head=8, n_embd=512),
    'large': GPTConfig(vocab_size=10000, block_size=1024, n_layer=12, n_head=12, n_embd=768),
}

print("Model sizes:")
for name, cfg in CONFIGS.items():
    model = GPT(cfg)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  {name}: {n_params/1e6:.2f}M parameters")
    del model

Model sizes:
  tiny: 2.10M parameters
  small: 14.66M parameters
  medium: 30.56M parameters
  large: 93.42M parameters


## Summary

### What You Learned

1. **GPT Architecture**: Decoder-only transformer
2. **Weight Tying**: Token embeddings = LM head weights
3. **Initialization**: GPT-2 style with scaled residual projections
4. **Parameter Distribution**: Most params in FFN and embeddings

### Next: Data Loading and Training

In Notebooks 06-07, we'll prepare data and train our GPT model!